# DS-471 Project – Stage 4: Model Training and Hyperparameter Optimization

Dataset: Breast Cancer Wisconsin (Diagnostic), 569 samples, 30 features (Malignant = 1, Benign = 0).

This notebook continues Stages 1–3: same 80/20 stratified split (455 train / 114 test), Stage 1 preprocessing (IQR outlier replacement with the median + StandardScaler, fitted inside each CV fold), and 5-fold stratified cross-validation on the training set. **The test set is not used here** (Stage 5).

The first code cell loads the dataset from scikit-learn (identical to the UCI file). If you already have the cleaned `df` from Stage 1, build `X` (30 feature columns) and `y` (M=1, B=0) from it instead.

Outputs: `nn_curves.png`, `validation_curves.png`, `stage4_results.json`.

In [ ]:
import json, time, warnings, copy
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_validate, validation_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss, accuracy_score

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"], "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.3})
OUT = ""   # figures/results are saved in the current folder

## data (Stages 1-3)

In [ ]:
# ------------------------------------------------------------------ data (Stages 1-3)
d = load_breast_cancer(as_frame=True)
X = d.data.copy()
y = 1 - d.target            # sklearn: 0=malignant -> flip so Malignant=1, Benign=0 (as in Stage 1)
X = X.loc[~X.duplicated()]; y = y.loc[X.index]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)      # 455 / 114 (Stage 3 setup)
print("train/test:", X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))
X_train, y_train = X_train.values, y_train.values

class IQRMedianReplacer(BaseEstimator, TransformerMixin):
    """Stage 1 outlier rule: values outside [Q1-1.5*IQR, Q3+1.5*IQR] -> feature median
    (bounds/medians are learned on the training part only)."""
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        q1, q3 = np.percentile(X, [25, 75], axis=0)
        iqr = q3 - q1
        self.lo_, self.hi_, self.med_ = q1 - 1.5 * iqr, q3 + 1.5 * iqr, np.median(X, axis=0)
        return self
    def transform(self, X):
        X = np.asarray(X, float)
        return np.where((X < self.lo_) | (X > self.hi_), self.med_, X)

def pipe(clf):
    return Pipeline([("outl", IQRMedianReplacer()), ("scale", StandardScaler()), ("clf", clf)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall",
           "f1": "f1", "roc_auc": "roc_auc"}

class EarlyStoppingMLP(ClassifierMixin, BaseEstimator):
    """Feed-forward NN (ReLU hidden layers, sigmoid/logistic output, Adam, mini-batch 32, L2 penalty alpha).
    15% of the training data is held out as a validation set; per-epoch training/validation loss and
    accuracy are stored in history_. With early_stopping=True training stops when the validation loss has
    not improved for `patience` epochs and the best weights are restored."""
    def __init__(self, hidden_layer_sizes=(32, 16), alpha=1e-4, learning_rate_init=1e-3, batch_size=32,
                 max_epochs=300, patience=20, early_stopping=True, val_fraction=0.15, random_state=SEED):
        self.hidden_layer_sizes = hidden_layer_sizes; self.alpha = alpha
        self.learning_rate_init = learning_rate_init; self.batch_size = batch_size
        self.max_epochs = max_epochs; self.patience = patience; self.early_stopping = early_stopping
        self.val_fraction = val_fraction; self.random_state = random_state
    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        Xt, Xv, yt, yv = train_test_split(X, y, test_size=self.val_fraction, stratify=y,
                                          random_state=self.random_state)
        clf = MLPClassifier(hidden_layer_sizes=tuple(self.hidden_layer_sizes), activation="relu", solver="adam",
                            alpha=self.alpha, learning_rate_init=self.learning_rate_init,
                            batch_size=self.batch_size, random_state=self.random_state)
        h = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        best, best_model, best_ep, wait = np.inf, None, 0, 0
        for ep in range(self.max_epochs):
            clf.partial_fit(Xt, yt, classes=[0, 1])
            ptr, pva = clf.predict_proba(Xt), clf.predict_proba(Xv)
            h["train_loss"].append(log_loss(yt, ptr)); h["val_loss"].append(log_loss(yv, pva))
            h["train_acc"].append(accuracy_score(yt, ptr.argmax(1))); h["val_acc"].append(accuracy_score(yv, pva.argmax(1)))
            if h["val_loss"][-1] < best - 1e-4:
                best, best_ep, wait = h["val_loss"][-1], ep + 1, 0
                best_model = copy.deepcopy(clf)
            else:
                wait += 1
            if self.early_stopping and wait >= self.patience:
                break
        self.model_ = best_model if self.early_stopping else clf
        self.history_, self.best_epoch_, self.classes_ = h, best_ep, np.array([0, 1])
        return self
    def predict_proba(self, X): return self.model_.predict_proba(np.asarray(X, float))
    def predict(self, X): return self.model_.predict(np.asarray(X, float))

models = {
    "Logistic Regression": (pipe(LogisticRegression(penalty="l2", solver="lbfgs", max_iter=5000)),
        {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100]}),
    "SVM (RBF)": (pipe(SVC(kernel="rbf", probability=True, random_state=SEED)),
        {"clf__C": [0.1, 1, 10, 100], "clf__gamma": ["scale", 0.001, 0.01, 0.1]}),
    "Random Forest": (pipe(RandomForestClassifier(random_state=SEED)),
        {"clf__n_estimators": [100, 200], "clf__max_depth": [None, 5, 10],
         "clf__min_samples_leaf": [1, 2, 4], "clf__max_features": ["sqrt", 0.5]}),
    "Neural Network (MLP)": (pipe(EarlyStoppingMLP()),
        {"clf__hidden_layer_sizes": [(16, 8), (32, 16), (64, 32)],
         "clf__alpha": [1e-4, 1e-3, 1e-2, 1e-1], "clf__learning_rate_init": [0.001, 0.01]}),
}
defaults = {  # untuned starting configuration (what Stage 3 describes)
    "Logistic Regression": pipe(LogisticRegression(max_iter=5000)),
    "SVM (RBF)": pipe(SVC(kernel="rbf", probability=True, random_state=SEED)),
    "Random Forest": pipe(RandomForestClassifier(n_estimators=100, random_state=SEED)),
    "Neural Network (MLP)": pipe(EarlyStoppingMLP(hidden_layer_sizes=(64, 32), alpha=1e-8, learning_rate_init=0.01,
                                                  max_epochs=150, early_stopping=False)),
}

def summarize(res):
    return {m: float(np.mean(res[f"test_{m}"])) for m in scoring} | \
           {m + "_sd": float(np.std(res[f"test_{m}"])) for m in scoring} | \
           {"train_accuracy": float(np.mean(res["train_accuracy"])),
            "train_f1": float(np.mean(res["train_f1"]))}

results = {"baseline": {}, "tuned": {}, "best_params": {}, "n_combos": {}, "time_s": {}}
best_est = {}
for name, (est, grid) in models.items():
    t = time.time()
    base = cross_validate(defaults[name], X_train, y_train, cv=cv, scoring=scoring,
                          return_train_score=True, n_jobs=1)
    results["baseline"][name] = summarize(base)
    gs = GridSearchCV(est, grid, cv=cv, scoring=scoring, refit="f1", return_train_score=True, n_jobs=1)
    gs.fit(X_train, y_train)
    i = gs.best_index_
    cvr = gs.cv_results_
    results["tuned"][name] = {m: float(cvr[f"mean_test_{m}"][i]) for m in scoring} | \
        {m + "_sd": float(cvr[f"std_test_{m}"][i]) for m in scoring} | \
        {"train_accuracy": float(cvr["mean_train_accuracy"][i]), "train_f1": float(cvr["mean_train_f1"][i])}
    results["best_params"][name] = {k.replace("clf__", ""): (list(v) if isinstance(v, tuple) else v)
                                    for k, v in gs.best_params_.items()}
    results["n_combos"][name] = len(cvr["params"])
    results["time_s"][name] = round(time.time() - t, 1)
    best_est[name] = gs.best_estimator_
    print(name, results["best_params"][name], "CV F1 %.4f acc %.4f" % (results["tuned"][name]["f1"], results["tuned"][name]["accuracy"]),
          "| baseline F1 %.4f" % results["baseline"][name]["f1"], f"({results['time_s'][name]}s)", flush=True)
    # keep top-3 configs for the appendix-style table
    order = np.argsort(cvr["rank_test_f1"])[:3]
    results.setdefault("top3", {})[name] = [
        {"params": {k.replace("clf__", ""): (list(v) if isinstance(v, tuple) else v) for k, v in cvr["params"][j].items()},
         "f1": float(cvr["mean_test_f1"][j]), "accuracy": float(cvr["mean_test_accuracy"][j])} for j in order]

## NN training / validation curves

In [ ]:
# ------------------------------------------------------------------ NN training / validation curves
def nn_hist(est_kwargs):
    H, bests = [], []
    for tr, va in cv.split(X_train, y_train):
        pre = Pipeline([("outl", IQRMedianReplacer()), ("scale", StandardScaler())]).fit(X_train[tr])
        m = EarlyStoppingMLP(**est_kwargs).fit(pre.transform(X_train[tr]), y_train[tr])
        H.append(m.history_); bests.append(int(np.argmin(m.history_["val_loss"])) + 1)
    L = min(len(h["val_loss"]) for h in H)
    arr = {k: np.array([h[k][:L] for h in H]) for k in H[0]}
    return arr, bests, [len(h["val_loss"]) for h in H]

bp = results["best_params"]["Neural Network (MLP)"]
nn_tuned = dict(hidden_layer_sizes=tuple(bp["hidden_layer_sizes"]), alpha=bp["alpha"],
                learning_rate_init=bp["learning_rate_init"], early_stopping=False, max_epochs=100)
nn_init = dict(hidden_layer_sizes=(64, 32), alpha=1e-8, learning_rate_init=0.01, early_stopping=False, max_epochs=150)
cur_init, b_init, n_init = nn_hist(nn_init)
cur_tuned, b_tuned, n_tuned = nn_hist(nn_tuned)

def stats(cur, bests, nrun):
    mval = cur["val_loss"].mean(0); i = int(mval.argmin())
    return {"epochs_plotted": int(mval.shape[0]), "epochs_run_per_fold": nrun, "best_epoch_per_fold": bests,
            "mean_best_epoch": float(np.mean(bests)), "epoch_of_min_mean_val_loss": i + 1,
            "train_loss_min_point": float(cur["train_loss"].mean(0)[i]), "val_loss_min_point": float(mval[i]),
            "train_acc_min_point": float(cur["train_acc"].mean(0)[i]), "val_acc_min_point": float(cur["val_acc"].mean(0)[i]),
            "train_loss_last": float(cur["train_loss"].mean(0)[-1]), "val_loss_last": float(mval[-1]),
            "train_acc_last": float(cur["train_acc"].mean(0)[-1]), "val_acc_last": float(cur["val_acc"].mean(0)[-1])}
results["nn_curves"] = {"initial_cfg": {"hidden_layer_sizes": [64, 32], "alpha": 1e-8, "learning_rate_init": 0.01, "epochs": 150},
                        "initial": stats(cur_init, b_init, n_init), "tuned": stats(cur_tuned, b_tuned, n_tuned)}
print(json.dumps(results["nn_curves"], indent=1), flush=True)

def plot_curves(curs, bests, titles, fname):
    fig, axes = plt.subplots(2, 2, figsize=(10, 7.2))
    for r, (cur, bst, ttl) in enumerate(zip(curs, bests, titles)):
        ep = np.arange(1, cur["val_loss"].shape[1] + 1)
        for c, (ktr, kva, ylab, kind) in enumerate([("train_loss", "val_loss", "Loss (binary cross-entropy)", "Loss"),
                                                    ("train_acc", "val_acc", "Accuracy", "Accuracy")]):
            ax = axes[r, c]
            for key, col, lab in [(ktr, "#1f77b4", "Training"), (kva, "#d62728", "Validation")]:
                m, sd = cur[key].mean(0), cur[key].std(0)
                ax.plot(ep, m, color=col, lw=2, label=lab)
                ax.fill_between(ep, m - sd, m + sd, color=col, alpha=0.15)
            ax.axvline(np.mean(bst), color="gray", ls="--", lw=1.2, label=f"Mean best epoch ({np.mean(bst):.0f})")
            ax.set_xlabel("Epoch"); ax.set_ylabel(ylab); ax.set_title(f"{ttl}: {kind}", fontsize=11)
            if kind == "Accuracy": ax.set_ylim(0.88, 1.005)
            ax.legend(loc="best", fontsize=9)
    fig.tight_layout(); fig.savefig(OUT + fname, dpi=200); plt.close(fig)

plot_curves([cur_init, cur_tuned], [b_init, b_tuned],
            ["(a) Initial NN (64-32, alpha≈0)", f"(b) Tuned NN ({bp['hidden_layer_sizes'][0]}-{bp['hidden_layer_sizes'][1]}, alpha={bp['alpha']})"], "nn_curves.png")

## validation curves for key hyperparameters

In [ ]:
# ------------------------------------------------------------------ validation curves for key hyperparameters
def vcurve(est, name, vals, ax, xlabel, logx=False, label_fn=None):
    tr, va = validation_curve(est, X_train, y_train, param_name=name, param_range=vals, cv=cv,
                              scoring="f1", n_jobs=1)
    xs = np.arange(len(vals)) if not logx else np.array(vals, float)
    ax.plot(xs, tr.mean(1), "o-", color="#1f77b4", label="Training F1")
    ax.fill_between(xs, tr.mean(1) - tr.std(1), tr.mean(1) + tr.std(1), color="#1f77b4", alpha=0.12)
    ax.plot(xs, va.mean(1), "s-", color="#d62728", label="CV validation F1")
    ax.fill_between(xs, va.mean(1) - va.std(1), va.mean(1) + va.std(1), color="#d62728", alpha=0.12)
    if logx: ax.set_xscale("log")
    else:
        ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in vals])
    ax.set_xlabel(xlabel); ax.set_ylabel("F1-score (malignant)"); ax.legend(fontsize=9)
    return {"values": [str(v) for v in vals], "train": tr.mean(1).round(4).tolist(), "val": va.mean(1).round(4).tolist()}

bpl = results["best_params"]
fig, axes = plt.subplots(2, 2, figsize=(10, 7.2))
vc = {}
vc["lr_C"] = vcurve(best_est["Logistic Regression"], "clf__C", [0.001, 0.01, 0.1, 1, 10, 100, 1000], axes[0, 0], "C (inverse regularisation strength)", logx=True)
axes[0, 0].set_title("(a) Logistic Regression – C", fontsize=11)
best_C = bpl["SVM (RBF)"]["C"]
vc["svm_gamma"] = vcurve(best_est["SVM (RBF)"], "clf__gamma", [0.0001, 0.001, 0.01, 0.1, 1, 10], axes[0, 1], "gamma (RBF kernel width)", logx=True)
axes[0, 1].set_title(f"(b) SVM (RBF) – gamma (C = {best_C})", fontsize=11)
vc["rf_depth"] = vcurve(best_est["Random Forest"], "clf__max_depth", [1, 2, 3, 5, 8, 12, None], axes[1, 0], "max_depth")
axes[1, 0].set_title("(c) Random Forest – max_depth", fontsize=11)
vc["nn_alpha"] = vcurve(best_est["Neural Network (MLP)"], "clf__alpha", [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 3], axes[1, 1], "alpha (L2 penalty)", logx=True)
axes[1, 1].set_title("(d) Neural Network – alpha", fontsize=11)
fig.tight_layout(); fig.savefig(OUT + "validation_curves.png", dpi=200); plt.close(fig)
results["validation_curves"] = vc

json.dump(results, open(OUT + "stage4_results.json", "w"), indent=1, default=str)
print("done")

## Summary tables

In [ ]:
import pandas as pd
rows = []
for m in results["tuned"]:
    t, b = results["tuned"][m], results["baseline"][m]
    rows.append({"Model": m, "Best params": results["best_params"][m],
                 "CV F1 (initial)": round(b["f1"], 4), "CV F1 (tuned)": round(t["f1"], 4),
                 "CV acc (tuned)": round(t["accuracy"], 4), "CV recall (tuned)": round(t["recall"], 4),
                 "CV ROC-AUC (tuned)": round(t["roc_auc"], 4),
                 "Train acc": round(t["train_accuracy"], 4)})
pd.DataFrame(rows)